In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
from urllib.parse import urljoin
import csv
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import os

In [ ]:
"""
LPG Price Scraper - petroldieselprice.com
Source: https://www.petroldieselprice.com/lpg-gas-cylinder-price
Run: python scrape_lpg_main.py
Requires: pip install requests beautifulsoup4 pandas
"""

URL = "https://www.petroldieselprice.com/lpg-gas-cylinder-price"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Manual state mapping (city -> state)
STATE_MAP = {
    "Agartala": "Tripura", "Ahmedabad": "Gujarat", "Aizawl": "Mizoram",
    "Amritsar": "Punjab", "Bengaluru": "Karnataka", "Bhopal": "Madhya Pradesh",
    "Bhubaneswar": "Odisha", "Chandigarh": "Chandigarh", "Chennai": "Tamil Nadu",
    "Dehradun": "Uttarakhand", "Diu": "Daman & Diu", "Fatehabad": "Haryana",
    "Gangtok": "Sikkim", "Goa Panaji": "Goa", "Guntur Amaravathi": "Andhra Pradesh",
    "Guwahati": "Assam", "Hyderabad": "Telangana", "Imphal": "Manipur",
    "Itanagar": "Arunachal Pradesh", "Jaipur": "Rajasthan", "Kavaratti": "Lakshadweep",
    "Kohima": "Nagaland", "Kolkata": "West Bengal", "Leh": "Ladakh",
    "Lucknow": "Uttar Pradesh", "Mumbai": "Maharashtra", "New Delhi": "Delhi",
    "Patna": "Bihar", "Pondicherry": "Puducherry", "Port Blair": "Andaman & Nicobar",
    "Raipur": "Chhattisgarh", "Ranchi": "Jharkhand", "Shillong": "Meghalaya",
    "Shimla": "Himachal Pradesh", "Silvassa": "Dadra & Nagar Haveli",
    "Srinagar": "Jammu & Kashmir", "Thiruvananthapuram": "Kerala",
}

def scrape():
    print(f"Fetching: {URL}")
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    print(f"Status: {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")

    # First table = city-wise price table
    table = soup.find("table", {"id": "customers"})
    if not table:
        print("ERROR: Table not found. Site structure may have changed.")
        return

    rows = table.find_all("tr")
    data = []

    for row in rows:
        cols = row.find_all("td")
        if len(cols) == 3:
            city = cols[0].get_text(strip=True)
            domestic_raw = cols[1].get_text(strip=True).replace("₹", "").replace("\u20b9", "").strip()
            commercial_raw = cols[2].get_text(strip=True).replace("₹", "").replace("\u20b9", "").strip()
            try:
                domestic = float(domestic_raw)
                commercial = float(commercial_raw)
            except ValueError:
                print(f"Skipping row (bad data): {city}")
                continue

            data.append({
                "City": city,
                "State": STATE_MAP.get(city, "Unknown"),
                "Domestic_14kg_INR": domestic,
                "Commercial_19kg_INR": commercial,
                "Comm_to_Dom_Ratio": round(commercial / domestic, 2),
                "Scraped_Date": datetime.today().strftime("%Y-%m-%d"),
                "Source": URL,
            })

    df = pd.DataFrame(data)
    out = "lpg_prices_india.csv"
    df.to_csv(out, index=False)

    print(f"\n--- Scraping Complete ---")
    print(df.to_string(index=False))
    print(f"\nTotal cities: {len(df)}")
    print(f"Domestic  | Min: ₹{df['Domestic_14kg_INR'].min()}  Max: ₹{df['Domestic_14kg_INR'].max()}  Avg: ₹{df['Domestic_14kg_INR'].mean():.2f}")
    print(f"Commercial| Min: ₹{df['Commercial_19kg_INR'].min()}  Max: ₹{df['Commercial_19kg_INR'].max()}  Avg: ₹{df['Commercial_19kg_INR'].mean():.2f}")
    print(f"\nSaved to: {out}")

if __name__ == "__main__":
    scrape()

Fetching: https://www.petroldieselprice.com/lpg-gas-cylinder-price
Status: 200

--- Scraping Complete ---
              City                State  Domestic_14kg_INR  Commercial_19kg_INR  Comm_to_Dom_Ratio Scraped_Date                                                   Source
          Agartala              Tripura             1073.5               2421.5               2.26   2026-04-23 https://www.petroldieselprice.com/lpg-gas-cylinder-price
         Ahmedabad              Gujarat              920.0               2098.0               2.28   2026-04-23 https://www.petroldieselprice.com/lpg-gas-cylinder-price
            Aizawl              Mizoram             1065.0               2551.0               2.40   2026-04-23 https://www.petroldieselprice.com/lpg-gas-cylinder-price
          Amritsar               Punjab              954.0               2185.0               2.29   2026-04-23 https://www.petroldieselprice.com/lpg-gas-cylinder-price
         Bengaluru            Karnataka          

In [ ]:
import requests
import csv
from bs4 import BeautifulSoup

def scrape_agartala_2026_prices(url, output_file):
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the page: {e}")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Target the historical data table
    tables = soup.find_all('table', id='customers')
    
    if len(tables) >= 2:
        target_table = tables[1]
        rows = target_table.find('tbody').find_all('tr')
        
        data_to_save = []
        
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                month_text = cols[0].get_text(strip=True)
                
                # Filter specifically for the year 2026
                if "2026" in month_text:
                    # Extract prices and trend (the text inside the span)
                    domestic_price = cols[1].contents[0].strip()
                    domestic_change = cols[1].find('span').get_text(strip=True) if cols[1].find('span') else "—"
                    
                    commercial_price = cols[2].contents[0].strip()
                    commercial_change = cols[2].find('span').get_text(strip=True) if cols[2].find('span') else "—"
                    
                    data_to_save.append([
                        month_text, 
                        domestic_price, domestic_change, 
                        commercial_price, commercial_change
                    ])

        # Save to CSV
        if data_to_save:
            headers = ['Month', '14.2kg Price', '14.2kg Change', '19kg Price', '19kg Change']
            with open(output_file, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(headers)
                writer.writerows(data_to_save)
            print(f"Successfully saved 2026 data to {output_file}")
        else:
            print("No data found for the year 2026.")
    else:
        print("Required table structure not found on page.")

# Configuration
URL = "https://petroldieselprice.com/lpg-gas-cylinder-price-in-Agartala"
FILENAME = "agartala_lpg_prices_2026.csv"

scrape_agartala_2026_prices(URL, FILENAME)

Successfully saved 2026 data to agartala_lpg_prices_2026.csv


In [ ]:
def get_all_city_links(main_url):
    """Finds all city-specific LPG links from the main page."""
    try:
        response = requests.get(main_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        city_links = []
        # The site usually lists city links with a specific URL pattern
        for a in soup.find_all('a', href=True):
            if 'lpg-gas-cylinder-price-in-' in a['href']:
                full_url = urljoin(main_url, a['href'])
                city_name = a.get_text(strip=True)
                city_links.append((city_name, full_url))
        
        # Remove duplicates
        return list(dict.fromkeys(city_links))
    except Exception as e:
        print(f"Error gathering city links: {e}")
        return []

def scrape_fluctuation_table(city_name, url):
    """Scrapes the historical fluctuation table from a specific city page."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all tables with id='customers'
        tables = soup.find_all('table', id='customers')
        
        # On these pages, the 2nd table (index 1) contains the 'Recent Revised' data
        if len(tables) < 2:
            return []

        target_table = tables[1]
        rows = target_table.find('tbody').find_all('tr')
        
        extracted_data = []
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                month = cols[0].get_text(strip=True)
                
                # Extract Price and Fluctuation (Change) for Domestic
                dom_price = cols[1].contents[0].strip()
                dom_change = cols[1].find('span').get_text(strip=True) if cols[1].find('span') else "—"
                
                # Extract Price and Fluctuation (Change) for Commercial
                com_price = cols[2].contents[0].strip()
                com_change = cols[2].find('span').get_text(strip=True) if cols[2].find('span') else "—"
                
                extracted_data.append([
                    city_name, month, 
                    dom_price, dom_change, 
                    com_price, com_change
                ])
        return extracted_data
    except Exception as e:
        print(f"Failed to scrape {city_name}: {e}")
        return []

def main():
    # Start at the main LPG directory page
    main_page_url = "https://petroldieselprice.com/lpg-gas-cylinder-price"
    output_csv = "india_cities_lpg_fluctuations.csv"
    
    print("Step 1: Gathering city URLs...")
    cities = get_all_city_links(main_page_url)
    print(f"Found {len(cities)} cities to scrape.")

    all_city_history = []
    
    # Step 2: Visit each city page
    for name, url in cities:
        print(f"Processing: {name}...")
        history = scrape_fluctuation_table(name, url)
        all_city_history.extend(history)
        
        # Delay to avoid being blocked by the server
        time.sleep(1) 

    # Step 3: Save to CSV
    if all_city_history:
        headers = [
            'City', 'Month', 
            'Domestic Price', 'Domestic Change', 
            'Commercial Price', 'Commercial Change'
        ]
        with open(output_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(all_city_history)
        print(f"\nSuccess! Data for all cities saved to {output_csv}")
    else:
        print("\nNo data collected. Please check the URL or your connection.")

if __name__ == "__main__":
    main()

Step 1: Gathering city URLs...
Found 37 cities to scrape.
Processing: Agartala...
Processing: Ahmedabad...
Processing: Aizawl...
Processing: Amritsar...
Processing: Bengaluru...
Processing: Bhopal...
Processing: Bhubaneswar...
Processing: Chandigarh...
Processing: Chennai...
Processing: Dehradun...
Processing: Diu...
Processing: Fatehabad...
Processing: Gangtok...
Processing: Goa Panaji...
Processing: Guntur Amaravathi...
Processing: Guwahati...
Processing: Hyderabad...
Processing: Imphal...
Processing: Itanagar...
Processing: Jaipur...
Processing: Kavaratti...
Processing: Kohima...
Processing: Kolkata...
Processing: Leh...
Processing: Lucknow...
Processing: Mumbai...
Processing: New Delhi...
Processing: Patna...
Processing: Pondicherry...
Processing: Port Blair...
Processing: Raipur...
Processing: Ranchi...
Processing: Shillong...
Processing: Shimla...
Processing: Silvassa...
Processing: Srinagar...
Processing: Thiruvananthapuram...

Success! Data for all cities saved to india_cities_

In [ ]:
import yfinance as yf
brent_data = yf.download('BZ=F', period='6mo')
brent_data['Close'].to_csv('brent_crude_prices.csv')


[*********************100%***********************]  1 of 1 completed


In [ ]:
# 1. Load your existing dataset
# Replace 'my_37_cities.csv' with your actual file name
df = pd.read_csv('india_cities_lpg_fluctuations.csv') 

# 2. Define the major Indian LPG import terminals and their coordinates (Lat, Lon)
lpg_ports = {
    'Kandla Port, Gujarat': (23.0132, 70.2185),
    'Mangalore Port, Karnataka': (12.9284, 74.8016),
    'Haldia Port, West Bengal': (22.0257, 88.0583),
    'Ennore Port, Tamil Nadu': (13.2500, 80.3300),
    'Pipavav Port, Gujarat': (20.9000, 71.5000)
}

# 3. Initialize the geocoder (Nominatim requires a unique user_agent name)
geolocator = Nominatim(user_agent="iit_bhilai_dsl251_project")

# 4. Function to find the nearest port and calculate the distance
def get_nearest_port_distance(city_name):
    try:
        # Get city coordinates by searching "City Name, India"
        location = geolocator.geocode(f"{city_name}, India")
        if not location:
            print(f"Could not find coordinates for {city_name}")
            return None, None
            
        city_coords = (location.latitude, location.longitude)
        
        # Calculate distance to all ports and find the closest one
        min_dist = float('inf')
        nearest_port = ""
        
        for port, port_coords in lpg_ports.items():
            dist = geodesic(city_coords, port_coords).kilometers
            if dist < min_dist:
                min_dist = dist
                nearest_port = port
                
        # IMPORTANT: Pause for 1 second between requests. 
        # The free Nominatim API will block you if you send requests too fast!
        time.sleep(1) 
        
        return round(min_dist, 2), nearest_port
        
    except Exception as e:
        print(f"Error processing {city_name}: {e}")
        return None, None

df[['Distance_to_Port_km', 'Nearest_Port']] = df.apply(
    lambda row: pd.Series(get_nearest_port_distance(row['City'])), axis=1
)

# 6. Save the enriched dataset
df.to_csv('cities_with_supply_distances.csv', index=False)
print("\ndone: cities_with_supply_distances.csv")


done: cities_with_supply_distances.csv


In [12]:
def save_page_to_html():
    url = "https://ppac.gov.in/consumption/products-wise"
    
    # Browser-like headers to avoid being blocked
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        print(f"Downloading {url}...")
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Check for errors

        # Save to file
        with open("ppac_data.html", "w", encoding="utf-8") as f:
            f.write(response.text)
        
        print("Success! File saved as 'ppac_data.html'")

    except Exception as e:
        print(f"Error saving file: {e}")

if __name__ == "__main__":
    save_page_to_html()

Success! File saved as 'ppac_data.html'


In [15]:
import pandas as pd
from bs4 import BeautifulSoup
import os

def extract_data_from_local_html():
    file_path = "ppac_data.html"
    
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return

    try:
        print(f"Reading {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')

        # Step 1: Find the actual table. 
        # PPAC usually puts data in a table with class 'table' or inside a specific div
        table = soup.find('table') 
        
        if not table:
            # If the AJAX didn't finish before you saved, the table might be missing.
            print("Error: No <table> element found in the file.")
            print("Make sure the table was visible on your screen before you saved the HTML.")
            return

        # Step 2: Convert the specific table found by BeautifulSoup into a DataFrame
        # We convert the table back to a string for Pandas
        df_list = pd.read_html(str(table))
        df = df_list[0]

        # Step 3: Handle Multi-level headers if they exist
        # If the first few rows are mostly 'NaN', it's a header issue
        if df.columns.nlevels > 1:
            df.columns = [' '.join(col).strip() for col in df.columns.values]

        # Step 4: Clean the data
        # Remove empty rows and columns
        df.dropna(how='all', axis=0, inplace=True)
        df.dropna(how='all', axis=1, inplace=True)

        # Step 5: Save
        output_file = "cleaned_consumption_data.csv"
        df.to_csv(output_file, index=False)
        
        print(f"Success! Extracted {len(df)} rows.")
        print("\n--- DATA PREVIEW ---")
        print(df.head(10)) # Show more rows to verify data content

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    extract_data_from_local_html()

Reading ppac_data.html...
Success! Extracted 0 rows.

--- DATA PREVIEW ---
Empty DataFrame
Columns: []
Index: []


C:\Users\kanis\AppData\Local\Temp\ipykernel_34372\1264003976.py:31: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_list = pd.read_html(str(table))


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_dynamic_table():
    url = "https://ppac.gov.in/consumption/products-wise"
    
    # Set up headless browser (runs in background)
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    driver = webdriver.Chrome(options=chrome_options)

    try:
        print("Opening browser and waiting for table to load...")
        driver.get(url)

        # Wait up to 20 seconds for the table rows (<tr>) to appear
        # The class 'table' is used by the AJAX injector in this site
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
        
        # Give it a few extra seconds to ensure all rows are injected
        time.sleep(3)

        # Get the rendered HTML
        html_content = driver.page_source
        
        # Use Pandas to parse the now-filled table
        dfs = pd.read_html(html_content)
        
        # Find the correct table (usually the one with actual data)
        # Filter for the table that contains "LPG" or "Diesel" to be sure
        final_df = None
        for df in dfs:
            if df.astype(str).apply(lambda x: x.str.contains('LPG|Diesel', case=False)).any().any():
                final_df = df
                break
        
        if final_df is not None:
            # Save to CSV
            output_file = "consumption_data.csv"
            final_df.to_csv(output_file, index=False)
            print(f"Success! Saved {len(final_df)} rows to {output_file}")
            print(final_df.head())
        else:
            print("Table found but it appears to be empty.")

    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_dynamic_table()

Opening browser and waiting for table to load...


C:\Users\kanis\AppData\Local\Temp\ipykernel_34372\1734097329.py:34: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html_content)


Success! Saved 23 rows to cleaned_consumption_data.csv
  Products April   May  June  July August September October November December  \
0      LPG  2545  2682  2554  2810   2833      2794    2871     2843     3067   
1  Naphtha   935   945   984   962   1065      1010     924      918      986   
2       MS  3449  3782  3522  3493   3544      3399    3665     3517     3555   
3      ATF   772   776   730   710    710       720     773      788      784   
4      SKO    26    39    41    36     34        37      43       43       42   

  January February March  Total Unnamed: 14  
0    3012     2822  2379  33212         NaN  
1    1057     1012   943  11741         NaN  
2    3511     3369  3779  42586         NaN  
3     827      764   807   9161         NaN  
4      38       38    44    460         NaN  


In [ ]:
import pandas as pd

def clean_petroleum_data(input_file, output_file):
    # 1. Load the CSV
    # We use low_memory=False in case the footer text confuses the type inference
    df = pd.read_csv(input_file)

    # 2. Find the "TOTAL" row
    # This searches the 'Products' column for the word 'TOTAL'
    if 'Products' in df.columns:
        # Get the index of the row where 'Products' is 'TOTAL'
        total_rows = df.index[df['Products'] == 'TOTAL'].tolist()
        
        if total_rows:
            stop_index = total_rows[0]
            # Slice the dataframe to keep everything from the start up to the TOTAL row
            df = df.iloc[:stop_index + 1]

    # 3. Clean up "Unnamed" columns
    # Your data has an 'Unnamed: 14' column which is likely empty
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    # 4. Save the cleaned version
    df.to_csv(output_file, index=False)
    print(f"Success! Cleaned data saved to {output_file}")

# Usage
clean_petroleum_data('cleaned_consumption_data.csv', 'final_cleaned_consumption_data.csv')

Success! Cleaned data saved to final_consumption_data.csv
